In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('olist_orders_dataset.csv')
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [2]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [3]:
orders['order_id'].nunique()

99441

In [4]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [5]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [6]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [8]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [9]:
orders["order_purchase_timestamp"].min()

Timestamp('2016-09-04 21:15:19')

In [10]:
orders["order_purchase_timestamp"].max()

Timestamp('2018-10-17 17:30:18')

In [11]:
orders["order_purchase_timestamp"].dt.year.value_counts().sort_index()

order_purchase_timestamp
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [12]:
sales_orders = orders[
    orders["order_status"] == "delivered"
].copy()

In [13]:
sales_orders[
    sales_orders["order_delivered_customer_date"]
    < sales_orders["order_purchase_timestamp"]
].shape[0]

0

In [14]:
sales_orders[
    sales_orders["order_delivered_customer_date"]
    < sales_orders["order_purchase_timestamp"]
].shape[0]

0

In [15]:
monthly_orders = (
    sales_orders
    .set_index("order_purchase_timestamp")
    .resample("ME")
    ["order_id"]
    .nunique()
)

monthly_orders

order_purchase_timestamp
2016-09-30       1
2016-10-31     265
2016-11-30       0
2016-12-31       1
2017-01-31     750
2017-02-28    1653
2017-03-31    2546
2017-04-30    2303
2017-05-31    3546
2017-06-30    3135
2017-07-31    3872
2017-08-31    4193
2017-09-30    4150
2017-10-31    4478
2017-11-30    7289
2017-12-31    5513
2018-01-31    7069
2018-02-28    6555
2018-03-31    7003
2018-04-30    6798
2018-05-31    6749
2018-06-30    6099
2018-07-31    6159
2018-08-31    6351
Freq: ME, Name: order_id, dtype: int64

In [16]:
sales_orders = sales_orders[
    (sales_orders["order_purchase_timestamp"] >= "2017-01-01") &
    (sales_orders["order_purchase_timestamp"] < "2018-09-01")
].copy()

In [17]:
order_items = pd.read_csv("olist_order_items_dataset.csv")

In [18]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [20]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [21]:
order_items.duplicated().sum()

np.int64(0)

In [22]:
order_items['order_id'].nunique()

98666

In [23]:
order_items.groupby('order_id')['order_item_id'].count().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: order_item_id, dtype: float64

In [24]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [29]:
sales_orders["order_id"].isin(
    order_items["order_id"]
).value_counts()

order_id
True    96211
Name: count, dtype: int64

In [30]:
sales = pd.merge(
    sales_orders,
    order_items,
    on="order_id",
    how="inner",
    validate="one_to_many"
)

In [31]:
total_revenue = round(sales["price"].sum(),2)
total_revenue

np.float64(13181027.13)

In [32]:
total_orders = sales["order_id"].nunique()
total_orders

96211

In [33]:
aov = round(total_revenue / total_orders,2)
aov

np.float64(137.0)

In [34]:
customers = pd.read_csv("olist_customers_dataset.csv")

In [35]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [36]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [37]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [38]:
customers.duplicated().sum()

np.int64(0)

In [39]:
customers['customer_id'].nunique()

99441

In [40]:
customers['customer_unique_id'].nunique()

96096

In [41]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [42]:
sales = pd.merge(
    sales,
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [43]:
sales["customer_unique_id"].isnull().sum()

np.int64(0)

In [44]:
sales["customer_state"].isnull().sum()

np.int64(0)

In [45]:
total_customers = sales[
    "customer_unique_id"
].nunique()

total_customers

93104

In [46]:
repeat_rate = round(1 - (total_customers / total_orders),2)

repeat_rate

0.03

In [47]:
products = pd.read_csv("olist_products_dataset.csv")
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [48]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [49]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [50]:
products.duplicated().sum()

np.int64(0)

In [51]:
category_translation = pd.read_csv("product_category_name_translation.csv")

category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [52]:
products_clean = pd.merge(
    products,
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

In [53]:
products_clean["product_category_name_english"].isnull().sum()

np.int64(623)

In [54]:
products_clean.loc[
    products_clean["product_category_name"].notna()
    & products_clean["product_category_name_english"].isna(),
    ["product_category_name", "product_category_name_english"]
].drop_duplicates()

,product_category_name,product_category_name_english
1628,pc_gamer,NaN
5821,portateis_cozinha_e_preparadores_de_alimentos,NaN


In [55]:
translation_fix = {
    "pc_gamer": "PC Gaming",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "Portable Kitchen & Food Preparation"
}

mask = (
    products_clean["product_category_name_english"].isna()
    & products_clean["product_category_name"].notna()
)

products_clean.loc[
    mask,
    "product_category_name_english"
] = products_clean.loc[
    mask,
    "product_category_name"
].map(translation_fix)

In [56]:
products_clean[
    "product_category_name_english"
] = products_clean[
    "product_category_name_english"
].fillna("Unknown")

In [57]:
products_clean[
    "product_category_name_english"
].isnull().sum()

np.int64(0)

In [58]:
sales = pd.merge(
    sales,
    products_clean[
        ["product_id", "product_category_name_english"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [60]:
sales.to_csv(
    "olist_executive_sales.csv",
    index=False
)